In [1]:
from together import Together
from datasets import load_dataset
import os, glob, json

In [2]:
base_dir = "/workspaces/OpenHands/evaluation/evaluation_outputs/outputs/princeton-nlp__SWE-bench-dev/CodeActAgent/"
run_name = "qwen480b_together_maxiter_100_N_18_golden_pass_baseline"
# swebench_hf_dataset = load_dataset("princeton-nlp/SWE-bench", split="dev")
swebench_hf_dataset = load_dataset("princeton-nlp/SWE-bench_Verified", split="test")




# import json

# output_path = "output_176_swebench_dev.jsonl"

# with open(output_path, "w") as f:
#     for instance_id in selected_ids:
#         hf_search_result = swebench_hf_dataset.filter(
#             lambda x: x["instance_id"] == instance_id
#         )[0]

#         to_write = {
#             "instance_id": instance_id,
#             "model_patch": hf_search_result["patch"],
#             "model_name_or_path": (
#                 "Qwen3-Coder-480B-A35B-Instruct-FP8_maxiter_100_N_v0.56.0-no-hint-run_1"
#             ),
#         }

#         f.write(json.dumps(to_write) + "\n")




In [3]:
# instance_id = "pydicom__pydicom-997" # "pydicom__pydicom-997" "pylint-dev__astroid-1030" "marshmallow-code__marshmallow-2123"

instance_id = "astropy__astropy-14365"

hf_search_result = swebench_hf_dataset.filter(lambda x: x["instance_id"] == instance_id)[0]

Filter:   0%|          | 0/500 [00:00<?, ? examples/s]

In [4]:
print(hf_search_result["problem_statement"])

ascii.qdp Table format assumes QDP commands are upper case
### Description

ascii.qdp assumes that commands in a QDP file are upper case, for example, for errors they must be "READ SERR 1 2" whereas QDP itself is not case sensitive and case use "read serr 1 2". 

As many QDP files are created by hand, the expectation that all commands be all-caps should be removed.

### Expected behavior

The following qdp file should read into a `Table` with errors, rather than crashing.
```
read serr 1 2 
1 0.5 1 0.5
```

### How to Reproduce

Create a QDP file:
```
> cat > test.qdp
read serr 1 2 
1 0.5 1 0.5
<EOF>

 > python
Python 3.10.9 (main, Dec  7 2022, 02:03:23) [Clang 13.0.0 (clang-1300.0.29.30)] on darwin
Type "help", "copyright", "credits" or "license" for more information.
>>> from astropy.table import Table
>>> Table.read('test.qdp',format='ascii.qdp')
Traceback (most recent call last):
...
    raise ValueError(f'Unrecognized QDP line: {line}')
ValueError: Unrecognized QDP line: read serr

In [5]:
print(hf_search_result["patch"])

diff --git a/astropy/io/ascii/qdp.py b/astropy/io/ascii/qdp.py
--- a/astropy/io/ascii/qdp.py
+++ b/astropy/io/ascii/qdp.py
@@ -68,7 +68,7 @@ def _line_type(line, delimiter=None):
     _new_re = rf"NO({sep}NO)+"
     _data_re = rf"({_decimal_re}|NO|[-+]?nan)({sep}({_decimal_re}|NO|[-+]?nan))*)"
     _type_re = rf"^\s*((?P<command>{_command_re})|(?P<new>{_new_re})|(?P<data>{_data_re})?\s*(\!(?P<comment>.*))?\s*$"
-    _line_type_re = re.compile(_type_re)
+    _line_type_re = re.compile(_type_re, re.IGNORECASE)
     line = line.strip()
     if not line:
         return "comment"
@@ -306,7 +306,7 @@ def _get_tables_from_qdp_file(qdp_file, input_colnames=None, delimiter=None):
 
             values = []
             for v in line.split(delimiter):
-                if v == "NO":
+                if v.upper() == "NO":
                     values.append(np.ma.masked)
                 else:
                     # Understand if number is int or float



In [41]:
print(hf_search_result["test_patch"])

diff --git a/test_requests.py b/test_requests.py
--- a/test_requests.py
+++ b/test_requests.py
@@ -157,6 +157,11 @@ def test_params_bytes_are_encoded(self):
                                    params=b'test=foo').prepare()
         assert request.url == 'http://example.com/?test=foo'
 
+    def test_binary_put(self):
+        request = requests.Request('PUT', 'http://example.com',
+                                   data=u"ööö".encode("utf-8")).prepare()
+        assert isinstance(request.body, bytes)
+
     def test_mixed_case_scheme_acceptable(self, httpbin):
         s = requests.Session()
         s.proxies = getproxies()



# Outcome-Level RCA

In [26]:
def find_patch_diff_glob(instance_id: str):
    prefix_dir = os.path.join(
        base_dir,
        run_name,
        "final_eval",
        "logs",
        "run_evaluation",
    )

    pattern = os.path.join(prefix_dir, "**", instance_id, "patch.diff")
    matches = glob.glob(pattern, recursive=True)
    if not matches:
        raise FileNotFoundError(f"Could not find {instance_id} generated patch file")
    # newest by mtime
    matches.sort(key=lambda p: os.stat(p).st_mtime, reverse=True)
    found_file = matches[0]

    if os.path.exists(found_file) and os.access(found_file, os.R_OK):
        return matches[0]
    else:
        raise FileNotFoundError(f"Could not find {instance_id} generated patch file")


def outcome_rca(instance_id, swebench_hf_dataset):

    hf_search_result = swebench_hf_dataset.filter(lambda x: x["instance_id"] == instance_id)[0]

    golden_patch = hf_search_result["patch"]
    task_description = hf_search_result["problem_statement"]

    generated_patch_file = find_patch_diff_glob(instance_id)

    with open(generated_patch_file, "r", encoding="utf-8") as f:
        generated_patch = f.read()

    outcome_rca_prompt_template = """
    You are an expert software engineer and code reviewer.
    Your task is to analyze why a generated code patch failed to fix a software issue,
    compared to the ground-truth developer patch from the SWE-bench dataset.

    ---
    ### Task Description
    {task_description}

    ---
    ### Ground Truth Patch (Correct Fix)
    {golden_patch}

    ---
    ### Generated Patch (Model's Attempt and it failed)
    {generated_patch}

    ---
    ### Instructions

    1. Carefully compare the generated patch and the ground-truth patch.
    2. Identify key semantic and structural differences — not just textual ones.
    3. Explain why the generated patch likely failed to fix the bug:
    - Incorrect logic?
    - Wrong function or file modified?
    - Partial or missing fix?
    - Breaking other functionality?
    - Misunderstanding of the test condition?
    4. Summarize the root cause of failure in plain language.
    5. Suggest concrete improvements that would help a model produce a better patch next time.

    ---
    ### Output Format

    Respond in this JSON format:

    {{
    "failure_reason": "<short summary of why the generated patch failed>",
    "difference_analysis": "<detailed explanation of key differences>",
    "improvement_suggestions": "<specific actionable suggestions for improvement>"
    }}
    Return ONLY a single valid JSON object with keys:
    failure_reason, difference_analysis, improvement_suggestions.
    No extra commentary, no code fences.
    """

    outcome_rca_prompt = outcome_rca_prompt_template.format(
        task_description=task_description,
        golden_patch=golden_patch,
        generated_patch=generated_patch
    )

    rca_output = together_inference(prompt=outcome_rca_prompt)
    return {
        "instance_id": instance_id,
        "task_description": task_description,
        "golden_patch": golden_patch,
        "generated_patch": generated_patch,
        "root_cause_analysis": json.loads(rca_output)
    }



def together_inference(prompt, model="Qwen/Qwen3-Coder-480B-A35B-Instruct-FP8"):
    client = Together(api_key="d8762e8d418b93a47f7cb11bdbb952b9cf5b3c0b4948b38fcfe70445c255ae1b")
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
    )
    return response.choices[0].message.content



def inference_time_reflection(task_description, generated_patch):
    reflection_prompt_template = """
    You are a reflective reviewer that inspects a proposed code patch against a given task description.
    You cannot execute code or run tests — instead, you reason about conceptual correctness, structural completeness,
    and consistency with the intended requirements.

    ---
    ### Task Description
    {task_description}

    ---
    ### Generated Patch (Model's Attempt and it failed)
    {generated_patch}

    ---
    ### Instructions and steps

    1. **Describe the patch**
    - Summarize in bullet points what this patch changes (files, functions, signatures, logic).

    2. **Check requirement coverage**
    - Compare your bullet points to the task description.
    - Does the patch modify all necessary files/functions?
    - Does it address the described issue *completely* and *accurately*?
    - Does it preserve consistency in function signatures, imports, and vectorization?
    - Are deprecated paths removed if the new logic replaces them?

    3. **Check structural & domain correctness**
    - Are the math/geometry/logical changes consistent with the described requirement?
    - Are output ranges and invariants respected?
    - Are shapes, vectorization, and numerical stability handled correctly?
    - Does the patch follow existing code style and avoid introducing ad-hoc scripts?

    4. **Decide outcome**
    - If everything looks correct and complete:
        Output a structured JSON approving the patch.
    - If something is wrong, incomplete, or inconsistent:
        Output a structured JSON rejecting the patch, explain why, and propose a plan (and optionally a new patch).

    ---
    ### OUTPUT FORMAT

    If the patch is good:
    {{
    "is_good_patch": true,
    "reason": "<why this satisfies the requirements completely and accurately>"
    }}

    If the patch is not good:
    {{
    "is_good_patch": false,
    "reason": "<what is missing or wrong>",
    "plan_to_fix": "<a concrete plan on what other files / algorithms / structures to change>"
    }}

    """

    reflection_prompt = reflection_prompt_template.format(
        task_description=task_description,
        generated_patch=generated_patch
    )

    reflection = together_inference(reflection_prompt)
    return json.loads(reflection)



In [29]:
instance_id = "pvlib__pvlib-python-1666"

rca = outcome_rca(
    instance_id=instance_id,
    swebench_hf_dataset=swebench_hf_dataset
)




In [30]:
reflection = inference_time_reflection(
    task_description=rca["task_description"],
    generated_patch=rca["generated_patch"],
)

In [31]:
reflection

{'is_good_patch': False,
 'reason': 'The generated patch attempts to modify the `_vf_row_sky_integ` function in `pvlib/bifacial/infinite_sheds.py` to replace `0.5 * (cosd(psi_t_shaded) + cst)` with `0.5 * (1 + cosd(surface_tilt + psi_t_shaded))`, but it fails to properly define or compute the `cst` variable that was previously used. The original code relies on a precomputed `cst = cosd(surface_tilt)`, which represents the cosine of the surface tilt angle and is part of the original view factor calculation. However, in the proposed fix, the logic changes fundamentally by removing `cst` and replacing it with a constant `1`, which alters the mathematical formulation. While the intent aligns with the issue description — to use the formula (1 + cos(surface_tilt + psi_t)) / 2 — the implementation introduces a structural inconsistency because `cst` is still referenced elsewhere in the original code. Additionally, the added test files are extraneous and not part of the core library structure; 

In [24]:
rca

{'instance_id': 'marshmallow-code__marshmallow-1229',
 'task_description': "`only` argument inconsistent between Nested(S, many=True) and List(Nested(S))\n```python\r\nfrom pprint import pprint\r\n\r\nfrom marshmallow import Schema\r\nfrom marshmallow.fields import Integer, List, Nested, String\r\n\r\n\r\nclass Child(Schema):\r\n    name = String()\r\n    age = Integer()\r\n\r\n\r\nclass Family(Schema):\r\n    children = List(Nested(Child))\r\n\r\n\r\nclass Family2(Schema):\r\n    children = Nested(Child, many=True)\r\n\r\nfamily = {'children':[\r\n    {'name': 'Tommy', 'age': 12},\r\n    {'name': 'Lily', 'age': 15},\r\n]}\r\n\r\npprint(Family( only=['children.name']).dump(family).data)\r\npprint(Family2( only=['children.name']).dump(family).data)\r\n```\r\nreturns\r\n```\r\n{'children': [{'age': 12, 'name': 'Tommy'}, {'age': 15, 'name': 'Lily'}]}\r\n{'children': [{'name': 'Tommy'}, {'name': 'Lily'}]}\r\n```\r\n\r\ntested with marshmallow 2.15.4\r\n\r\nThe same applies to `exclude` arg

In [25]:
for key in rca["root_cause_analysis"]:
    print(key)
    print(rca["root_cause_analysis"][key])
    print("-"*100)

failure_reason
The generated patch incorrectly modifies the schema-level logic for handling nested field options, rather than addressing the core issue at the field level where List and Nested fields interact.
----------------------------------------------------------------------------------------------------
difference_analysis
The ground-truth patch correctly identifies that the inconsistency lies in how List and Dict fields handle their nested Nested field's 'only' and 'exclude' attributes. It modifies the __init__ and _bind_to_schema methods of List, Tuple, and Dict fields to properly propagate these attributes to/from their contained Nested fields. In contrast, the generated patch attempts to fix the issue by modifying schema-level normalization logic in BaseSchema._normalize_nested_options and __apply_nested_option, which is the wrong layer of abstraction. The generated patch also introduces debug prints and changes the target field for option application, but fails to address th

In [20]:
print(rca["golden_patch"])

diff --git a/pvlib/bifacial/infinite_sheds.py b/pvlib/bifacial/infinite_sheds.py
--- a/pvlib/bifacial/infinite_sheds.py
+++ b/pvlib/bifacial/infinite_sheds.py
@@ -6,66 +6,9 @@
 import pandas as pd
 from pvlib.tools import cosd, sind, tand
 from pvlib.bifacial import utils
-from pvlib.shading import masking_angle
 from pvlib.irradiance import beam_component, aoi, haydavies
 
 
-def _vf_ground_sky_integ(surface_tilt, surface_azimuth, gcr, height,
-                         pitch, max_rows=10, npoints=100, vectorize=False):
-    """
-    Integrated view factor to the sky from the ground underneath
-    interior rows of the array.
-
-    Parameters
-    ----------
-    surface_tilt : numeric
-        Surface tilt angle in degrees from horizontal, e.g., surface facing up
-        = 0, surface facing horizon = 90. [degree]
-    surface_azimuth : numeric
-        Surface azimuth angles in decimal degrees east of north
-        (e.g. North = 0, South = 180, East = 90, West = 270).
-        ``su

In [21]:
print(rca["generated_patch"])

diff --git a/pvlib/bifacial/infinite_sheds.py b/pvlib/bifacial/infinite_sheds.py
index 7c83b13..958bbdf 100644
--- a/pvlib/bifacial/infinite_sheds.py
+++ b/pvlib/bifacial/infinite_sheds.py
@@ -144,14 +144,14 @@ def _vf_row_sky_integ(f_x, surface_tilt, gcr, npoints=100):
     # shaded portion
     x = np.linspace(0, f_x, num=npoints)
     psi_t_shaded = masking_angle(surface_tilt, gcr, x)
-    y = 0.5 * (cosd(psi_t_shaded) + cst)
+    y = 0.5 * (1 + cosd(surface_tilt + psi_t_shaded))
     # integrate view factors from each point in the discretization. This is an
     # improvement over the algorithm described in [2]
     vf_shade_sky_integ = np.trapz(y, x, axis=0)
     # unshaded portion
     x = np.linspace(f_x, 1., num=npoints)
     psi_t_unshaded = masking_angle(surface_tilt, gcr, x)
-    y = 0.5 * (cosd(psi_t_unshaded) + cst)
+    y = 0.5 * (1 + cosd(surface_tilt + psi_t_unshaded))
     vf_noshade_sky_integ = np.trapz(y, x, axis=0)
     return vf_shade_sky_integ, vf_noshade_sky_inte

# Docker deep dive debug

In [10]:
import tarfile
import json
from pathlib import PurePosixPath


def read_file_from_tar(tar, member_name):
    """Read a specific file from a tar, return decoded text or None."""
    try:
        f = tar.extractfile(member_name)
        return f.read().decode() if f else None
    except KeyError:
        return None



def check_docker_libraries(tar_path):
    # ---- LOAD TOP-LEVEL TAR (manifests + layer tars) ----
    with tarfile.open(tar_path, "r") as tar:
        manifest = json.loads(tar.extractfile("manifest.json").read())
        layers = manifest[0]["Layers"]

        python_packages = set()
        apt_entries = None
        conda_packages = set()

        # ---- ITERATE THROUGH LAYERS ----
        for layer_name in layers:
            layer_file = tar.extractfile(layer_name)

            # Open nested layer.tar
            with tarfile.open(fileobj=layer_file) as layer_tar:

                for member in layer_tar.getmembers():
                    path = PurePosixPath(member.name)

                    # ---- Python packages ----
                    if "site-packages" in path.parts or "dist-packages" in path.parts:
                        if path.suffix == ".dist-info":
                            python_packages.add(path.name.replace(".dist-info", ""))

                    # ---- Conda ----
                    if "conda-meta" in path.parts and path.suffix == ".json":
                        conda_packages.add(path.stem)

                    # ---- APT packages ----
                    if path == PurePosixPath("var/lib/dpkg/status"):
                        file_text = layer_tar.extractfile(member).read().decode()
                        apt_entries = file_text


    # ----- PRINT RESULTS -----
    print("\n=== Python packages ===")
    for pkg in sorted(python_packages):
        print(pkg)

    print("\n=== Conda packages ===")
    for c in sorted(conda_packages):
        print(c)

    print("\n=== APT packages ===")
    if apt_entries:
        for line in apt_entries.splitlines():
            if line.startswith("Package:"):
                print(line.split("Package:")[1].strip())
    else:
        print("(none or dpkg not used)")


In [11]:

runtime_tar_path = "/workspaces/Openhands/runtime_dockers/swebench_dev/pvlib__pvlib-python-1518/oh_v0.56.0_qoa4fsbtt5ffktz5_78wywo64b09wmysu.tar"
baseline_tar_path = "/workspaces/Openhands/swebench_dockers_for_eval/swebench_dockers/dev/docker_images/pvlib__pvlib-python-1518__latest.tar"

In [12]:
check_docker_libraries(baseline_tar_path)


=== Python packages ===
Bottleneck-1.4.2
Brotli-1.0.9
Cython-3.0.12
MarkupSafe-3.0.2
PySocks-1.7.1
PyYAML-6.0.2
Send2Trash-1.8.3
Shapely-1.8.5.post1
Sphinx-4.5.0
alabaster-0.7.16
anyio-4.9.0
archspec-0.2.1
argon2_cffi-23.1.0
argon2_cffi_bindings-21.2.0
arrow-1.3.0
asttokens-3.0.0
async_lru-2.0.5
attrs-25.3.0
autocommand-2.2.2
babel-2.17.0
backports.tarfile-1.2.0
beautifulsoup4-4.13.4
bleach-6.2.0
boltons-23.0.0
certifi-2023.11.17
certifi-2025.4.26
cffi-1.16.0
cffi-1.17.1
cftime-1.6.4.post1
charset_normalizer-2.0.4
charset_normalizer-3.4.2
comm-0.2.2
conda-23.11.0
conda_content_trust-0.2.0
conda_libmamba_solver-23.12.0
conda_package_handling-2.2.0
conda_package_streaming-0.9.0
contourpy-1.3.0
coverage-7.8.0
cryptography-41.0.7
cycler-0.12.1
debugpy-1.8.14
decorator-5.2.1
defusedxml-0.7.1
distro-1.8.0
docutils-0.15.2
ephem-4.2
exceptiongroup-1.2.2
executing-2.2.0
fastjsonschema-2.21.1
flake8-7.2.0
fonttools-4.57.0
fqdn-1.5.1
future-1.0.0
h11-0.16.0
h5py-3.13.0
html5lib-1.1
httpcore-1.0.

In [13]:
check_docker_libraries(runtime_tar_path)


=== Python packages ===
Bottleneck-1.4.2
Brotli-1.0.9
Cython-3.0.12
Deprecated-1.2.18
Farama_Notifications-0.0.4
GitPython-3.1.44
MarkupSafe-3.0.2
PyGithub-2.6.1
PyJWT-2.10.1
PyNaCl-1.5.0
PySocks-1.7.1
PyYAML-6.0.2
QtPy-2.4.3
SecretStorage-3.3.3
Send2Trash-1.8.3
Shapely-1.8.5.post1
Sphinx-4.5.0
aiohappyeyeballs-2.6.1
aiohttp-3.12.14
aiosignal-1.4.0
alabaster-0.7.16
annotated_types-0.7.0
anthropic-0.59.0
anyio-4.11.0
anyio-4.9.0
archspec-0.2.1
argon2_cffi-23.1.0
argon2_cffi-25.1.0
argon2_cffi_bindings-21.2.0
arrow-1.3.0
asttokens-3.0.0
async_lru-2.0.5
attrs-25.3.0
authlib-1.6.0
autocommand-2.2.2
babel-2.17.0
backports.tarfile-1.2.0
bashlex-0.18
beautifulsoup4-4.13.4
bidict-0.23.1
binaryornot-0.4.4
bleach-6.2.0
boltons-23.0.0
boto3-1.39.13
botocore-1.39.13
brotli-1.1.0
browsergym_core-0.13.3
build-1.2.2.post1
build-1.3.0
cachecontrol-0.14.3
cachetools-5.5.2
certifi-2023.11.17
certifi-2025.10.5
certifi-2025.4.26
cffi-1.16.0
cffi-1.17.1
cffi-2.0.0
cftime-1.6.4.post1
chardet-5.2.0
charset_